In [1]:
!wget -q https://dl.google.com/linux/direct/google-chrome-stable_current_amd64.deb
!apt-get update -qq
!apt-get install -y -qq ./google-chrome-stable_current_amd64.deb
!pip install -q selenium
!pip install undetected-chromedriver
!pip install dash plotly

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libatk1.0-data.
(Reading database ... 118212 files and directories currently installed.)
Preparing to unpack .../00-libatk1.0-data_2.36.0-3build1_all.deb ...
Unpacking libatk1.0-data (2.36.0-3build1) ...
Selecting previously unselected package libatk1.0-0:amd64.
Preparing to unpack .../01-libatk1.0-0_2.36.0-3build1_amd64.deb ...
Unpacking libatk1.0-0:amd64 (2.36.0-3build1) ...
Selecting previously unselected package libatspi2.0-0:amd64.
Preparing to unpack .../02-libatspi2.0-0_2.44.0-3_amd64.deb ...
Unpacking libatspi2.0-0:amd64 (2.44.0-3) ...
Selecting previously unselected package libatk-bridge2.0-0:amd64.
Preparing to unpack .../03-libatk-bridge2.0-0_2.38.0-3_amd64.deb ...
Unpacking libatk-bridge2.0-0:amd64 (2.38.0-3) ...
Selecting previously unselected pack

In [2]:
import plotly.express as px
from dash import Dash, dcc, html
import re
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter
import pandas as pd
from bs4 import BeautifulSoup
import undetected_chromedriver as uc
from selenium.webdriver.support.ui import WebDriverWait


class ZoonRuParser:

    catalog_url = "https://zoon.ru/msk/vet/type/gruming-salon/?v%5Brating_4%5D%5Bf%5D=4&search_query_form=1$0"

    card_selector = "li.minicard-item.js-results-item"
    salon_link_selector = "a.title-link"
    name_selector = "a.title-link"
    price_row_selector = "tr.service-list-item"
    service_name_selector = "td.service-name"
    service_price_selector = "td.service-price"


    def __init__(self, wait_timeout: int = 15):
        self.wait_timeout = wait_timeout
        self.driver = None
        self.wait = None
        self.salons = []
        self.df_salons = None
    def start(self) -> None:
        options = uc.ChromeOptions()
        options.add_argument("--headless=new")
        options.add_argument("--no-sandbox")
        options.add_argument("--disable-dev-shm-usage")
        options.add_argument("--window-size=1920,1080")
        self.driver = uc.Chrome(
            options=options,
            browser_executable_path="/usr/bin/google-chrome",
            version_main=148,
        )
        self.wait = WebDriverWait(self.driver, self.wait_timeout)


    def close(self) -> None:
        if self.driver:
            self.driver.quit()
            self.driver = None

    def __enter__(self):
        self.start()
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        self.close()

    @staticmethod
    def _extract_numbers(text: str) -> list[int]:
        raw = re.findall(r"\d[\d\s\u00a0]*", text)
        return [int(re.sub(r"\D", "", n)) for n in raw if n.strip()]

    @staticmethod
    def _extract_coords(card):
        lat_raw = card.get("data-lat")
        lon_raw = card.get("data-lon")
        try:
            return float(lat_raw), float(lon_raw)
        except (TypeError, ValueError):
            return None, None

    def collect_salon_links(self) -> list[dict]:
        url1 = "https://zoon.ru/msk/vet/type/gruming-salon/"
        url2 =  "/?v%5Brating_4%5D%5Bf%5D=4&search_query_form=1$0"
        salons = []

        for page in range(1, 5):
            page_url = f"{url1}page-{page}{url2}"
            self.driver.get(page_url)
            time.sleep(3)
            soup = BeautifulSoup(self.driver.page_source, "lxml")
            cards = soup.select(self.card_selector)

            if not cards:
                print(f"каталог закончился на {page} странице")
                break
            for card in cards:
                link_tag = card.select_one(self.salon_link_selector)
                if not link_tag:
                    continue
                url = link_tag.get("href", "")
                if not url:
                    continue
                url_with_price = f"{url}price/"
                name_tag = card.select_one(self.name_selector)
                lat, lon = self._extract_coords(card)
                salons.append({
                    "salon": name_tag.get_text(strip=True) if name_tag else None,
                    "url": url_with_price,
                    "lat": lat,
                    "lon": lon,
                })
            self.salons = salons
            self.df_salons = pd.DataFrame(salons)
            self.df_salons.to_csv("zoon_salons.csv", index=False, encoding="utf-8-sig")
            print(f"страница {page}: сохранено, всего {len(salons)} салонов")
        return salons

    def build_dashboard(self):
        df_salons = self.df_salons
        df_map = df_salons.dropna(subset=["lat", "lon"])

        fig = px.density_mapbox(
            df_map,
            lat="lat",
            lon="lon",
            radius=20,
            center=dict(lat=55.75, lon=37.62),
            zoom=9,
            mapbox_style="open-street-map",
            hover_name="salon",
        )
        fig.update_layout(margin=dict(l=0, r=0, t=30, b=0))

        app = Dash(__name__)
        app.layout = html.Div([
            html.H2("плотность груминг-салонов конкурентов"),
            html.P(f"салонов всего: {len(df_map)}"),
            dcc.Graph(figure=fig),
        ])
        app.run(jupyter_mode="inline")
        return app



with ZoonRuParser() as pr:
    pr.collect_salon_links()
    pr.build_dashboard()



страница 1: сохранено, всего 30 салонов
страница 2: сохранено, всего 60 салонов
страница 3: сохранено, всего 90 салонов
страница 4: сохранено, всего 120 салонов
Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit


In [ ]:
# в коллабе это не запускается, я запустил в вс коде и там получилось, ниже код от туда

%pip install pandas plotly dash
import plotly.express as px
from dash import Dash, dcc, html
imoprt pandas as pd
df_salons = pd.read_csv("zoon_salons.csv")
df_map = df_salons.dropna(subset=["lat", "lon"])

fig = px.density_mapbox(
    df_map, lat="lat", lon="lon", radius=20,
    center=dict(lat=55.75, lon=37.62), zoom=9,
    mapbox_style="open-street-map", hover_name="name",
)
fig.update_layout(margin=dict(l=0, r=0, t=30, b=0))

app = Dash(__name__)
app.layout = html.Div([
    html.H2("плотность груминг-салонов конкурентов", style={"color": "deeppink"}),
    html.P(f"всего салонов: {len(df_map)}", style={"color": "deeppink"}),
    dcc.Graph(figure=fig),
])

app.run(debug=True)